In [1]:
from build123d import *
from ocp_vscode import *


wall = 100
clearance=50


In [2]:
class ModelSpool(BasePartObject):
    spool_diameter = 2740
    spool_width = 1340
    cutout_diameter = 980

    def __init__(
        self,
        rotation: RotationLike = (0, 0, 0),
        align: tuple[Align, Align, Align] = (Align.CENTER, Align.CENTER, Align.CENTER),
        mode: Mode = Mode.ADD,
    ):
        with BuildPart() as part:
            with BuildSketch() as hub:
                Circle(radius=(self.cutout_diameter / 2) + 100)
                Circle(radius=(self.cutout_diameter / 2), mode=Mode.SUBTRACT)
            extrude(amount=(self.spool_width/2)-140)

            with BuildSketch(part.faces().group_by(Axis.Z)[-1]) as side:
                Circle(radius=self.spool_diameter / 2)
                Circle(radius=self.cutout_diameter / 2, mode=Mode.SUBTRACT)
            # Extrude the sketch to create the spool model
            extrude(amount=70)
            mirror(part.part, about=Plane.XY)
        self.label = "ModelSpool"
        super().__init__(part.part, rotation=rotation, align=align, mode=mode)


In [3]:
class SpoolLegs(BaseSketchObject):
    def __init__(
        self,
        rotation: float = 0, 
        align: tuple[Align, Align] = (Align.CENTER, Align.CENTER),
        mode: Mode = Mode.ADD,
        length: int = ModelSpool.spool_diameter,
        height: int = 300,
    ):
        with BuildSketch() as sketch:
            SlotCenterToCenter(center_separation=length, height=height)
            Circle(radius=ModelSpool.spool_diameter/2, mode=Mode.SUBTRACT, align=(Align.CENTER, Align.MIN))
        self.label = "SpoolLegs"

        super().__init__(sketch.sketch, rotation=rotation, align=align, mode=mode)



In [4]:
with BuildPart() as part:
    wall_width = 100
    with BuildSketch() as layer0:
        Circle(radius=(ModelSpool.spool_diameter / 2) + 130)
        Circle(radius=(ModelSpool.spool_diameter / 2) - 130, mode=Mode.SUBTRACT)
        for r in range(0, 360, 60):
            Rectangle(height=2500, width=200, mode=Mode.ADD, rotation=r)
        Circle(radius=(ModelSpool.cutout_diameter / 2) + 250)
        Circle(radius=(ModelSpool.cutout_diameter / 2) - 75, mode=Mode.SUBTRACT)
        # with Locations((0, -ModelSpool.spool_diameter/2-130, 0)):
        #     SpoolLegs(align=(Align.CENTER, Align.MIN))
    extrude(amount=wall_width)

    ### Hub Stack
    with BuildSketch(part.faces().group_by(Axis.Z)[-1]) as layer2:
        Circle(radius=(ModelSpool.spool_diameter / 2) + 130)
        Circle(radius=(ModelSpool.spool_diameter / 2) + 30, mode=Mode.SUBTRACT)
        Circle(radius=ModelSpool.cutout_diameter / 2 - 25)
        Circle(radius=(ModelSpool.cutout_diameter / 2) - 75, mode=Mode.SUBTRACT)
        with PolarLocations(radius=(ModelSpool.spool_diameter / 2+100), count=1, start_angle=45):
            Rectangle(height=50, width=200, mode=Mode.SUBTRACT)
    extrude(amount=ModelSpool.spool_width/2+100)

    ## Hub Holder with cutouts
    with BuildSketch(part.faces().group_by(Axis.Z)[-1]) as layer3:
        Circle(radius=(ModelSpool.spool_diameter / 2) + 130)
        Circle(radius=(ModelSpool.spool_diameter / 2) + 30, mode=Mode.SUBTRACT)
        Circle(radius=ModelSpool.cutout_diameter / 2 - 25)
        Circle(radius=(ModelSpool.cutout_diameter / 2) - 75, mode=Mode.SUBTRACT)
        for r in range(0, 360, 60):
            Rectangle(height=ModelSpool.cutout_diameter + 100, width=100, mode=Mode.SUBTRACT, rotation=r)
        with PolarLocations(radius=(ModelSpool.spool_diameter / 2+100), count=1, start_angle=45):
            Rectangle(height=50, width=200, mode=Mode.SUBTRACT)
    extrude(amount=ModelSpool.spool_width/2)

    ## Spool Retention Clip
    with Locations((0, 0, ModelSpool.spool_width+200)):
        with BuildSketch(layer2.faces().group_by(Axis.Z)[-1]) as clip:
            Circle(radius=ModelSpool.cutout_diameter / 2 + 25)
            Circle(radius=(ModelSpool.cutout_diameter / 2) - 125, mode=Mode.SUBTRACT)
            for r in range(0, 360, 60):
                Rectangle(height=ModelSpool.cutout_diameter + 200, width=100, mode=Mode.SUBTRACT, rotation=r)
        extrude(amount=100.0, taper=45.0)
        with BuildSketch(layer2.faces().group_by(Axis.Z)[-1]) as cutout:
            Circle(radius=ModelSpool.cutout_diameter / 2 - 75)
        extrude(amount=100.0, mode=Mode.SUBTRACT)

    ### Base Stabilizer
    with Locations((0, -ModelSpool.spool_diameter/2-50, ModelSpool.spool_width/2+wall_width/2+50)):
        with BuildSketch(Plane.XZ) as stabilizer:
            RectangleRounded(height=ModelSpool.spool_width+wall_width+100, width = ModelSpool.spool_diameter, radius=200)
            RectangleRounded(height=ModelSpool.spool_width-wall_width*2, width = ModelSpool.spool_diameter-200, radius=100, mode=Mode.SUBTRACT)
        extrude(amount=200)

    part.label = "WirewrapDispenser"


In [5]:

part = part.part.scale(0.0254)
# show(part, ModelSpool().part)
ms = ModelSpool()
ms = ms.moved(Location((0, 0, (ms.spool_width/2+20) * 0.0254)))
ms = ms.scale(0.0254)
# print(ms)
# print(part)
show(part)
# show(ms, part, colors=["cyan", "yellow"])
export_stl(part, "/Volumes/SSD 990 2TB/Documents/stl/wirewrap_dispenser.stl")


Using port 3939
+


True